<div style="text-align:center; padding:20px 0"><img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/></div>

# AfriCare Support Analytics## Notebook 4 — Dashboard Power BI & Storytelling> **Prérequis** : Notebooks 2, 3 et 5 complétés. Les fichiers CSV exportés (données nettoyées + scores ML) doivent être disponibles.| | ||---|---|| **Niveau** | Avancé || **Outils** | Power BI Desktop || **Durée estimée** | 4h à 5h |> 💡 **Ce notebook est un guide de conception reproductible.** En le suivant pas à pas, tu produiras le dashboard exactement tel qu'il apparaît dans le rapport de référence (4 pages avec navigation par onglets horizontaux AfriCare Support).### Objectif businessTransformer les analyses SQL et ML en un dashboard décisionnel **4 pages** permettant à M. Kouamé de piloter : santé opérationnelle · performance SLA · classement agents · alertes ML préventives.

---## 1. Sources de données (4 fichiers)### Fichiers à importer dans Power BI| Fichier CSV | Renommer en | Rôle | Volume ||---|---|---|---|| `support_clean_analytics.csv` | `fact_support_analytics` | Fait (tickets) | 15 287 lignes || `tickets_risque_scores.csv` | *(garder ce nom)* | Fait analytique (ML) | 3 058 lignes || `agents.csv` | `dim_agents` | Dimension agents | 12 lignes || `categories.csv` | `dim_categories` | Dimension catégories | 10 lignes |### Import1. Power BI Desktop → **Obtenir des données → Texte/CSV**2. Mode **Import** pour chaque fichier3. Vérifier les types en Power Query avant de charger (notamment `sla_breach` et `in_backlog` en **Nombre entier** — 0/1)4. Renommer les tables selon le tableau ci-dessus dans le panneau Champs

---## 2. Désactiver Auto Date/Time (obligatoire)Avant toute manipulation du modèle :**Fichier → Options → Chargement des données (Fichier actuel) → DÉCOCHER "Date/heure automatique pour le fichier actuel"**Sans cette étape, Power BI crée des tables `LocalDateTable_*` parasites qui polluent le modèle et empêchent `PREVIOUSMONTH` / `SAMEPERIODLASTYEAR` de fonctionner correctement.

---## 3. Modèle de données — Schéma en étoile### Architecture```                    dim_agents (12 lignes)                         |dim_categories --- fact_support_analytics --- Calendrier  (10 lignes)        (15 287 lignes)         (1 ligne/jour)                         |                 tickets_risque_scores                   (3 058 lignes)```### Relations à configurer (4 relations)| Table (N) | Colonne | Table (1) | Colonne | Cardinalité | Direction ||---|---|---|---|---|---|| `fact_support_analytics` | `agent_id` | `dim_agents` | `agent_id` | N→1 | Single || `fact_support_analytics` | `category_id` | `dim_categories` | `category_id` | N→1 | Single || `fact_support_analytics` | `created_at` (date) | `Calendrier` | `Date` | N→1 | Single || `tickets_risque_scores` | `ticket_id` | `fact_support_analytics` | `ticket_id` | 1→1 | Single |### ⚠️ Si `created_at` contient une heureDans Power Query, créer une colonne calculée `date_created` = `Date.From([created_at])` pour relier proprement à `Calendrier[Date]`. Puis lier `date_created → Calendrier[Date]` au lieu de `created_at`.

---## 4. Table CalendrierModélisation → **Nouvelle table** → coller :```daxCalendrier =ADDCOLUMNS(    CALENDAR(DATE(2022,1,1), DATE(2024,12,31)),    "Annee",         YEAR([Date]),    "Mois_Num",      MONTH([Date]),    "Mois_Nom",      FORMAT([Date], "MMM", "fr-FR"),    "Mois_Nom_Long", FORMAT([Date], "MMMM", "fr-FR"),    "Annee_Mois",    FORMAT([Date], "YYYY-MM"),    "Trimestre",     "T" & QUARTER([Date]),    "Semaine",       WEEKNUM([Date]),    "Jour_Semaine",  FORMAT([Date], "dddd", "fr-FR"),    "Est_Weekend",   IF(WEEKDAY([Date],2) >= 6, 1, 0))```**Puis marquer comme table de dates** : clic droit `Calendrier` → *Marquer comme table de dates* → colonne `Date`.

---## 5. Table _Mesures (placeholder)Modélisation → **Nouvelle table** → coller :```dax_Mesures = {BLANK()}```Puis **masquer la colonne `Value`** (clic droit → Masquer). La table sert uniquement à héberger les mesures dans des dossiers d'affichage.

---## 6. Design system — AfriCare Support### ⚠️ Spécificité vs autres rapports DPLCe rapport utilise un **design clair** (fond blanc/gris pâle) — à la différence des dashboards HotelChain (or/noir) ou E-commerce (navy). Chaque onglet porte une couleur dédiée qui signale visuellement la section en cours.### Palette AfriCare| Usage | Couleur | Hex ||---|---|---|| Fond principal | Blanc cassé | `#FFFFFF` || Fond section | Gris très clair | `#F5F5F7` || **Onglet Overview** (actif) | Violet | `#5B21B6` || **Onglet Performance SLA** (actif) | Orange | `#F97316` || **Onglet Agents** (actif) | Vert | `#10B981` || **Onglet Alertes ML** (actif) | Rouge | `#EF4444` || Onglet inactif | Gris texte | `#6B7280` || Performance positive / Résolu | Vert | `#10B981` || Alerte / Escaladé | Rouge | `#EF4444` || Vigilance / Backlog | Orange | `#F59E0B` || En cours | Bleu | `#3B82F6` || Fermé / Neutre | Gris moyen | `#9CA3AF` || Jaune (alerte ML niveau 2) | Jaune | `#FBBF24` || Texte principal | Gris foncé | `#111827` || Texte secondaire | Gris moyen | `#6B7280` |### Typographie| Usage | Police | Taille ||---|---|---|| Titres pages (serif) | **Georgia** regular | 22-26px || Navigation onglets | Segoe UI medium | 14px || Valeurs KPI | Segoe UI bold | 32-38px || Labels KPI | Segoe UI regular | 11-12px |### Principes- Fond clair global → contraste élevé pour lecture rapide en environnement bureau- Chaque carte KPI : bordure supérieure colorée 3px selon la nature du KPI- Les bars par catégorie de breach sont colorées **par seuil** (rouge ≥ 50%, orange 30-50%, vert < 30%) — voir mesure `Couleur Barre Breach`

---## 7. Architecture de navigation — Onglets horizontaux### ⚠️ Différence avec les autres rapports DPLIci la navigation est **horizontale en haut de page** (pas de sidebar latérale). Chaque onglet est un bouton "Navigation de page" avec sa couleur dédiée quand il est actif.### Bandeau supérieur (présent sur les 4 pages)**Zone gauche** : Logo avatar AfriCare (48×48) + texte `AfriCare Support` en Segoe UI bold 18px.**Zone centrale** : 4 onglets de navigation, alignés horizontalement.| Ordre | Label | Couleur active | Page cible ||---|---|---|---|| 1 | Overview | Violet `#5B21B6` | Page 1 || 2 | Performance SLA | Orange `#F97316` | Page 2 || 3 | Agents | Vert `#10B981` | Page 3 || 4 | Alertes ML | Rouge `#EF4444` | Page 4 |**Style de l'onglet actif** : fond de la couleur dédiée, texte blanc bold, forme arrondie (border-radius 8px), ombre portée légère.**Style des onglets inactifs** : fond transparent, texte gris `#6B7280`, pas de bordure.**Zone droite** : 2 slicers date range avec icône calendrier — `01/01/2022` et `31/12/2024` (voir section 8).### Implémentation Power BIPour chaque onglet :1. Insertion → **Bouton** → texte de l'onglet2. Volet Format → Action → **Type : Navigation de page** → cible = page correspondante3. Pour l'état actif : mettre en forme le bouton **uniquement sur la page cible** avec la couleur dédiée4. Dupliquer les 4 boutons sur les 4 pages (copy/paste)

---## 8. Slicers globaux### Un seul groupe : Date range**Dans le bandeau supérieur droit** des 4 pages :| Slicer | Champ | Style | Valeurs par défaut ||---|---|---|---|| **Date début** | `Calendrier[Date]` | **Between** (mode plage) ou **Entre** | 01/01/2022 || **Date fin** | `Calendrier[Date]` | Idem | 31/12/2024 |Les 2 champs date peuvent être présentés en **un seul slicer Between** (2 champs date affichés côte à côte) ou en **2 slicers Date après/avant** accolés.### Style- Fond blanc, bordure grise fine `#E5E7EB`- Icône calendrier à droite de chaque champ- Texte date en Segoe UI regular 13px### SynchronisationClic droit sur chaque slicer → *Synchroniser les segments* → cocher les 4 pages (visible ET filtre).> 💡 **Pas d'autre slicer global** dans ce rapport. Les filtres spécifiques par page (canal, catégorie, agent) peuvent être ajoutés au volet Filtres mais ne sont pas exposés en slicer visible.

---## 9. Page 1 — Overview**Titre** : `Vue d'ensemble — Support AfriCare` (Georgia 24px gris foncé)**Onglet actif** : Overview (violet)### Ligne 1 : 6 KPI cardsChaque carte : fond blanc, bordure supérieure colorée 3px, valeur en Segoe UI bold 38px colorée, pastille variation en-dessous.| # | Label | Valeur principale | Complément | Couleur top ||---|---|---|---|---|| 1 | Tickets total | `[Total Tickets]` → **15 287** | ▲ +24,9% vs N-1 | 🔘 Gris `#9CA3AF` || 2 | Taux SLA breach | `[Taux SLA Breach]` → **47,3%** | ▲ 0,0pp vs N-1 | 🔴 Rouge `#EF4444` || 3 | Backlog | `[Taux Backlog]` → **7,4%** | `[Nb Tickets Backlog]` = 1 126 tickets | 🟠 Orange `#F59E0B` || 4 | Délai 1ère réponse | `[First Response Moy]` → **36,8h** | ▼ +0,1h vs N-1 | 🔴 Rouge `#EF4444` || 5 | CSAT moyen | `[CSAT Moyen]` → **4,12** | ★★★★☆ /5 | 🟢 Vert `#10B981` || 6 | Réouverture | `[Taux Reouverture]` → **7,7%** | `[Nb Reouvertures]` = 1 181 tickets | 🔘 Gris `#9CA3AF` |> Les pastilles de variation utilisent les mesures `SLA Breach vs N-1`, `Tickets vs N-1`, `Delai vs N-1` avec flèches UNICHAR.### Ligne 2 : 2 visuels côte à côte**Gauche — Évolution taux SLA breach (%)** (courbe lissée)- Visuel : **Graphique en courbes**- Axe X : `Calendrier[Mois_Nom]` (janv à déc)- Axe Y : `[Taux SLA Breach]`- Couleur : rouge `#EF4444`, ligne épaisse 3px, courbe lissée- Ligne constante horizontale rose pointillée à 30% (cible SLA)- Format : 0,0%**Droite — Répartition des statuts** (donut)- Visuel : **Graphique en anneau**- Légende : `fact_support_analytics[statut]`- Valeurs : `[Total Tickets]`- Data labels : % intérieur (ligne de rappel pour Résolu)- Palette : Résolu vert `#10B981` · Escaladé rouge `#EF4444` · En cours bleu `#3B82F6` · Backlog orange `#F59E0B` · Fermé gris `#9CA3AF`- Résultat : Résolu 52,78% · Escaladé 21,74% · En cours 15% · Backlog 7,37% · Fermé ~3%### Ligne 3 : 2 visuels côte à côte**Gauche — Taux SLA breach par catégorie (%)** (bar chart horizontal)- Visuel : **Histogramme à barres** trié DESC- Axe Y : `dim_categories[nom_categorie]`- Axe X : `[Taux SLA Breach]`- **Couleur par formule** : utiliser la mesure `Couleur Barre Breach` (rouge ≥ 50%, orange 30-50%, vert < 30%)- Ligne constante verticale rouge pointillée à 50% (seuil contractuel)- Data labels activés (format 0,0%)- Top observé : Demande info 68,6% · Facturation 59,4% · Remboursement 52,6% · Livraison retardée 51,6% · Qualité produit 51,1% · Panne technique 39,4% · Problème paiement 37,8% · Fraude signalée 25,5% · Escalade juridique 23,2% · Accès compte 20,7%**Droite — Tickets par pays** (bar chart horizontal)- Visuel : **Histogramme à barres** trié DESC- Axe Y : `fact_support_analytics[pays]` (Top 5)- Axe X : `[Total Tickets]`- Couleur unique : violet `#5B21B6`- Format labels : 4k / 3k / 2k (milliers)- Top 5 : CI 4k · Sénégal 3k · France 2k · Maroc 2k · Ghana 2k### Alerte contractuelle (en bas)Bandeau conditionnel de largeur complète (sous les 2 visuels de droite) :- Visuel : **Carte** avec mesure `Alerte Contractuelle`- Bordure rouge 2px, fond blanc- Titre : `Alerte contractuelle` (rouge bold)- Texte : *"Billing SLA strict : **3 catégories** dépassent 50% de breach. Risque de pénalités contractuelles."*- **Visible uniquement** si `[Nb Categories >50% Breach] > 0` (formatage conditionnel sur la visibilité/couleur de fond)

---## 10. Page 2 — Performance SLA**Titre** : `Performance SLA — par catégorie & canal` (Georgia 24px)**Onglet actif** : Performance SLA (orange)### Ligne 1 : 4 KPI cards + 1 jauge| # | Label | Valeur | Complément | Couleur top ||---|---|---|---|---|| 1 | Breach global | `[Taux SLA Breach]` → **47,3%** | Objectif : 30% | 🔴 Rouge || 2 | Breach SLA strict | `[Breach SLA Strict]` → **42,9%** | Risque contractuel (texte rouge) | 🔴 Rouge || 3 | Dépassement moyen | `[Depassement Moyen h]` → **+121h** | Sur tickets en breach | 🟠 Orange || 4 | Ratio SLA moyen | `[Ratio SLA Moyen]` → **1,00×** | Résolution / SLA contractuel | 🟠 Orange |**À droite des 4 cartes — Jauge circulaire % breach vs objectif**- Visuel : **Jauge** (natif) ou **KPI circulaire** custom- Valeur : `[Taux SLA Breach]` → 47,3%- Min : 0% · Max : 100% · Cible : 30%- Zones colorées : 0-30% vert, 30-50% orange, 50-100% rouge- Titre : "% breach vs objectif 30%"### Ligne 2 : 2 visuels côte à côte**Gauche — Taux SLA breach par catégorie (%)** (bar chart horizontal)- **Identique à celui de la Page 1** (même mesure `Couleur Barre Breach`, même tri DESC, même ligne cible verticale à 50%)- Réplication volontaire : permet la lecture autonome de cette page**Droite — Répartition des statuts** (donut)- **Identique à celui de la Page 1** (même palette, même mesure)- Version légèrement plus grande (met en évidence le 3,12% Fermé visible)### Ligne 3 : 2 visuels côte à côte**Gauche — Heatmap Catégorie × Mois** (tableau matrice)- Visuel : **Matrice** native- Lignes : `dim_categories[nom_categorie]`- Colonnes : `Calendrier[Mois_Num]` (janv à déc)- Valeurs : `[Taux SLA Breach]` (format 0,0%)- **Formatage conditionnel sur la couleur de fond** : règle *Par dégradé* → Min 20% vert `#86EFAC` · Milieu 45% jaune `#FEF08A` · Max 70% rouge `#FCA5A5`- Supprimer totaux lignes/colonnes pour garder uniquement les cellules- Lecture : on voit les catégories problématiques (Demande info, Facturation) en rouge quasi tous les mois**Droite — Taux SLA breach par canal (%)** (bar chart vertical)- Visuel : **Histogramme à colonnes**- Axe X : `fact_support_analytics[canal]`- Axe Y : `[Taux SLA Breach]`- Couleur unique : violet foncé `#5B21B6`- Data labels activés (format 0,0%)- Résultat : Téléphone 48,0% · Chat 47,9% · App mobile 47,4% · Web form 46,6% · Email 46,2%- Insight : les 5 canaux sont quasi homogènes → le canal n'est PAS un facteur différenciant

---## 11. Page 3 — Agents**Titre** : `Performance agents — classement & matrice` (Georgia 24px)**Onglet actif** : Agents (vert)### Layout en 2 blocs#### Gauche : Tableau Classement agents**Visuel : Table** (native) avec 7 colonnes et 12 lignes.| Colonne | Champ / Mesure | Formatage ||---|---|---|| Agent | `dim_agents[nom_agent]` | Segoe UI regular || Tier | `dim_agents[tier]` | **Badge coloré** : Tier 1 gris clair, Tier 2 vert clair, Tier 3 violet clair (formatage conditionnel couleur de fond) || Rg SLA | `[Rank SLA Agent]` | Bleu centré || Breach | `[Taux SLA Breach]` | **Texte coloré** : rouge ≥ 50%, orange 40-50%, vert < 40% || CSAT | `[CSAT Moyen]` | Format 0,0 || Écart | `[Ecart CSAT]` | Format 0,00 (différence vs moyenne globale) || Segment | `[Segment Agent]` | Badge jaune clair "Coach" (uniforme ici) |Les 12 agents observés (triés alphabétiquement) :| Agent | Tier | Breach | CSAT | Écart ||---|---|---|---|---|| Aissatou Ba | Tier 1 | 48,3% | 3,9 | -0,07 || Aminata Diallo | Tier 1 | 43,4% | 4,2 | 0,00 || Aya Touré | Tier 2 | 46,3% | 4,7 | -0,10 || Fatou Sow | Tier 2 | 39,9% | 4,5 | -0,10 || Ibrahim Coulibaly | Tier 2 | 42,2% | 4,3 | -0,07 || Jean-Marc Dubois | Tier 3 | 40,0% | 4,8 | -0,12 || Kofi Mensah | Tier 1 | 53,6% | 3,8 | -0,10 || **Moussa Kone** | Tier 1 | **60,1%** | 3,5 | -0,14 || Nadia Benhaddou | Tier 1 | 48,3% | 4,0 | -0,14 || Olivier Martin | Tier 3 | 44,8% | 4,6 | -0,08 || **Ramatou Diaby** | Tier 1 | **55,1%** | 3,6 | -0,08 || Samuel Acheampong | Tier 2 | 45,1% | 4,4 | -0,02 |3 agents Tier 1 en alerte (> 50% breach) : Moussa Kone · Ramatou Diaby · Kofi Mensah.#### Droite : Matrice performance — Breach vs CSAT**Visuel : Nuage de points (Scatter)**- Axe X : `[Taux SLA Breach]` (de 40% à 60%)- Axe Y : `[CSAT Moyen]` (de 3,4 à 4,8)- Détails : `dim_agents[nom_agent]`- **Légende (couleur)** : `dim_agents[tier]` → Tier 1 gris pâle · Tier 2 vert pâle · Tier 3 violet pâle- Taille bulle : valeur fixe ou `[Nb Tickets Agent]`- Opacité 60% pour voir les superpositions> 💡 **Pas de lignes de quadrant dans ce rapport** (contrairement au notebook original). La lecture se fait visuellement : les bulles en haut-gauche (faible breach + CSAT élevé) = top performers Tier 3 ; les bulles en bas-droite (fort breach + CSAT bas) = coaching prioritaire.

---## 12. Page 4 — Alertes ML**Titre** : `Alertes ML — Tickets à risque détectés` (Georgia 24px)**Onglet actif** : Alertes ML (rouge)### Ligne 1 : 5 KPI cardsChaque carte a un fond légèrement teinté de sa couleur de criticité.| # | Label | Valeur | Complément | Couleur ||---|---|---|---|---|| 1 | Alertes ROUGE | `[Tickets Risque Rouge]` → **314** | Intervention immédiate | 🔴 Rouge `#EF4444` (fond rosé) || 2 | Alertes ORANGE | `[Tickets Risque Orange]` → **1 542** | À surveiller | 🟠 Orange `#F59E0B` (fond orangé pâle) || 3 | Alertes JAUNE | `[Tickets Risque Jaune]` → **931** | Attention | 🟡 Jaune `#FBBF24` (fond jaune pâle) || 4 | Score risque moyen | `[Score Risque Moy]` → **0,50** | Sur tickets alertés | 🟣 Violet `#8B5CF6` || 5 | Recall modèle | `[Recall Modele]` → **79,6%** | Objectif > 75% ✓ (checkmark vert) | 🟢 Vert `#10B981` |### Ligne 2 : Tableau tickets + Distribution alertes**Gauche (largeur 60%) — Liste des tickets alertés (triés par score DESC)**- Visuel : **Table** native- Pré-filtrée sur `niveau_alerte` contient "ROUGE" (filtre de page)- Trié par `score_risque` DESC| Colonne | Champ | Formatage ||---|---|---|| Ticket | `tickets_risque_scores[ticket_id]` | Regular || Pays | `fact_support_analytics[pays]` | Regular || Canal | `fact_support_analytics[canal]` | Regular || Agent | `dim_agents[nom_agent]` | Regular || Score | `tickets_risque_scores[score_risque]` | Format 0,00 || Priorite | `fact_support_analytics[priorite]` | Entier || Niveau Alerte | `tickets_risque_scores[niveau_alerte]` | **Badge rouge** texte blanc |Exemples observés (top 5) :- TKT014857 · Cameroun · Email · Jean-Marc Dubois · 0,84 · 5 · ROUGE - Intervention immediate- TKT012691 · CI · Téléphone · Moussa Kone · 0,82 · 4 · ROUGE- TKT013209 · Ghana · Téléphone · Jean-Marc Dubois · 0,81 · 5 · ROUGE- TKT013797 · Maroc · Chat · Moussa Kone · 0,81 · 4 · ROUGE- TKT014745 · France · Web form · Moussa Kone · 0,81 · 5 · ROUGE**Droite haut (largeur 40%) — Distribution des niveaux d'alerte** (bar vertical)- Visuel : **Histogramme à colonnes**- Axe X : `tickets_risque_scores[niveau_alerte]`- Axe Y : `[Nb Tickets Risque]` (comptage)- **Couleur par catégorie** :  - ORANGE - Surveiller → orange pâle `#FDBA74`  - JAUNE - Attention → jaune doré `#EAB308`  - ROUGE - Intervention immediate → rouge pâle `#FCA5A5`  - VERT - Normal → vert `#10B981`- Data labels activés- Ordre observé : ORANGE 1542 · JAUNE 931 · ROUGE 314 · VERT 271### Ligne 3 : Top agents + Jauge score**Droite milieu — Top agents — alertes ROUGE** (bar horizontal)- Visuel : **Histogramme à barres** trié DESC- Axe Y : `dim_agents[nom_agent]` (Top 5)- Axe X : `[Tickets Risque Rouge]`- Couleur : rouge `#EF4444` pour les 3 premiers, orange `#F59E0B` pour les suivants- Top 5 : Moussa Kone 66 · Ramatou Diaby 52 · Aissatou Ba 36 · Kofi Mensah 34 · Jean-Marc Dubois 26**Droite bas — Score risque moyen** (jauge horizontale)- Visuel : **Jauge linéaire** custom ou **Bar chart 1 catégorie**- Échelle 0 à 1 avec 4 zones colorées :  - 0 à 0,25 : vert `#10B981` (label "0 — Vert")  - 0,25 à 0,45 : jaune `#EAB308` (label "0.25 — Jaune")  - 0,45 à 0,70 : orange `#F59E0B` (label "0.45 — Orange")  - 0,70 à 1 : rouge `#EF4444` (label "0.70 — Rouge")- Marqueur vertical noir positionné à `[Score Risque Moy]` = 0,50- Label sous la jauge : "**Score moyen : 0,50**" en Segoe UI bold

---## 13. Mesures DAX — Table `_Mesures`### 📂 1. KPIs de base (10 mesures)```daxTotal Tickets = COUNTROWS(fact_support_analytics)Taux SLA Breach = DIVIDE(    CALCULATE(COUNTROWS(fact_support_analytics), fact_support_analytics[sla_breach] = 1),    [Total Tickets])Nb Tickets Backlog = CALCULATE(COUNTROWS(fact_support_analytics), fact_support_analytics[in_backlog] = 1)Taux Backlog = DIVIDE([Nb Tickets Backlog], [Total Tickets])Taux Escalade = DIVIDE(    CALCULATE(COUNTROWS(fact_support_analytics), fact_support_analytics[statut] = "Escalade"),    [Total Tickets])CSAT Moyen = AVERAGEX(    FILTER(fact_support_analytics, NOT(ISBLANK(fact_support_analytics[csat]))),    fact_support_analytics[csat])First Response Moy = AVERAGE(fact_support_analytics[first_response_heures])Resolution Moy = AVERAGE(fact_support_analytics[resolution_heures])Nb Reouvertures = CALCULATE(COUNTROWS(fact_support_analytics), fact_support_analytics[reouvert] = 1)Taux Reouverture = DIVIDE([Nb Reouvertures], [Total Tickets])```### 📂 2. KPIs SLA avancés — Page 2 (3 mesures)```dax-- Catégories soumises à SLA strict (paramètre Billing)Breach SLA Strict = DIVIDE(    CALCULATE(        COUNTROWS(fact_support_analytics),        fact_support_analytics[sla_breach] = 1,        dim_categories[sla_strict] = 1    ),    CALCULATE(        COUNTROWS(fact_support_analytics),        dim_categories[sla_strict] = 1    ))-- Dépassement moyen en heures, uniquement sur tickets en breachDepassement Moyen h = CALCULATE(    AVERAGE(fact_support_analytics[depassement_heures]),    fact_support_analytics[sla_breach] = 1)-- Ratio résolution réelle / SLA contractuelRatio SLA Moyen = AVERAGEX(    fact_support_analytics,    DIVIDE(fact_support_analytics[resolution_heures], fact_support_analytics[sla_heures]))```

---## 14. Mesures DAX — Variations vs N-1 (4 mesures)Les pastilles de variation sur la Page 1 comparent l'année en cours à N-1 (année précédente).```dax-- Variation du nombre de tickets (%)Tickets vs N-1 = VAR _curr = [Total Tickets]VAR _prev = CALCULATE([Total Tickets], SAMEPERIODLASTYEAR(Calendrier[Date]))VAR _pct = DIVIDE(_curr - _prev, _prev)VAR _fmt = FORMAT(_pct, "+0.0%;-0.0%")RETURNIF(_pct >= 0, UNICHAR(9650) & " " & _fmt & " vs N-1",              UNICHAR(9660) & " " & _fmt & " vs N-1")-- Variation du taux SLA breach (en points de %)SLA Breach vs N-1 = VAR _curr = [Taux SLA Breach]VAR _prev = CALCULATE([Taux SLA Breach], SAMEPERIODLASTYEAR(Calendrier[Date]))VAR _delta = (_curr - _prev) * 100VAR _fmt = FORMAT(_delta, "+0.0;-0.0") & "pp"RETURNIF(_delta >= 0, UNICHAR(9650) & " " & _fmt & " vs N-1",                UNICHAR(9660) & " " & _fmt & " vs N-1")-- Variation du délai de 1ère réponse (en heures)Delai vs N-1 = VAR _curr = [First Response Moy]VAR _prev = CALCULATE([First Response Moy], SAMEPERIODLASTYEAR(Calendrier[Date]))VAR _delta = _curr - _prevVAR _fmt = FORMAT(_delta, "+0.0;-0.0") & "h"RETURNIF(_delta >= 0, UNICHAR(9650) & " " & _fmt & " vs N-1",                UNICHAR(9660) & " " & _fmt & " vs N-1")-- Nombre de catégories dépassant 50% de breach (pour alerte contractuelle)Nb Categories >50% Breach = COUNTROWS(    FILTER(        VALUES(dim_categories[nom_categorie]),        [Taux SLA Breach] > 0.50    ))-- Texte dynamique pour le bandeau d'alerteAlerte Contractuelle = VAR _nb = [Nb Categories >50% Breach]RETURNIF(    _nb > 0,    "Billing SLA strict : " & _nb & " categories depassent 50% de breach. Risque de penalites contractuelles.",    BLANK())```

---## 15. Mesures DAX — Alertes ML (Page 4) — 6 mesures```dax-- Utilisation de SEARCH pour matcher "ROUGE -- Intervention immediate" quel que soit le suffixeTickets Risque Rouge = CALCULATE(    COUNTROWS(tickets_risque_scores),    SEARCH("ROUGE", tickets_risque_scores[niveau_alerte], 1, 0) > 0)Tickets Risque Orange = CALCULATE(    COUNTROWS(tickets_risque_scores),    SEARCH("ORANGE", tickets_risque_scores[niveau_alerte], 1, 0) > 0)Tickets Risque Jaune = CALCULATE(    COUNTROWS(tickets_risque_scores),    SEARCH("JAUNE", tickets_risque_scores[niveau_alerte], 1, 0) > 0)Tickets Risque Vert = CALCULATE(    COUNTROWS(tickets_risque_scores),    SEARCH("VERT", tickets_risque_scores[niveau_alerte], 1, 0) > 0)-- Score moyen sur les tickets alertés uniquement (exclure VERT)Score Risque Moy = CALCULATE(    AVERAGE(tickets_risque_scores[score_risque]),    SEARCH("VERT", tickets_risque_scores[niveau_alerte], 1, 0) = 0)-- Recall du modèle ML (doit être > 75% pour validation métier)-- Valeur issue du Notebook 5, stockée en colonne calculée ou en paramètre What-IfRecall Modele = 0.796   -- ou lire depuis tickets_risque_scores si colonne présente```### 📂 Couleur Barre Breach (formatage conditionnel Pages 1 et 2)```daxCouleur Barre Breach = VAR _tx = [Taux SLA Breach]RETURNSWITCH(TRUE(),    _tx >= 0.50, "#EF4444",   -- Rouge    _tx >= 0.30, "#F59E0B",   -- Orange    "#10B981"                 -- Vert)```**Utilisation** : Visuel bar chart → Format → *Couleurs des données* → `fx` → *Par formule* → sélectionner `Couleur Barre Breach`.

---## 16. Mesures DAX — Agents (Page 3) — 3 mesures```dax-- Classement agent par taux breach croissant (rang 1 = meilleur SLA)-- ALL(agent_id) est obligatoire pour forcer l'évaluation sur TOUS les agentsRank SLA Agent = RANKX(    ALL(dim_agents[agent_id]),    CALCULATE([Taux SLA Breach]),    ,    ASC)-- Écart du CSAT agent par rapport à la moyenne globaleEcart CSAT = VAR _agentCsat = [CSAT Moyen]VAR _globalCsat = CALCULATE([CSAT Moyen], ALL(dim_agents))RETURN _agentCsat - _globalCsat-- Segment agent (texte du badge) — dans ce rapport tous à "Coach"-- La logique peut être raffinée : Top / Coach / Urgent selon Écart + BreachSegment Agent = VAR _ecart = [Ecart CSAT]VAR _breach = [Taux SLA Breach]RETURNSWITCH(TRUE(),    _breach >= 0.55, "Urgent",    _breach >= 0.40 && _ecart < 0, "Coach",    _ecart >= 0 && _breach < 0.40, "Top",    "Coach")-- Nombre de tickets par agent (pour taille bulle scatter)Nb Tickets Agent = CALCULATE(    COUNTROWS(fact_support_analytics),    USERELATIONSHIP(fact_support_analytics[agent_id], dim_agents[agent_id]))```

---## 17. Checklist de validation### Import & modèle- [ ] 4 fichiers importés et renommés (fact + tickets_risque_scores + 2 dim)- [ ] Auto Date/Time désactivé- [ ] Table `Calendrier` créée et marquée comme table de dates- [ ] Table `_Mesures` créée (colonne Value masquée)- [ ] 4 relations actives, aucun chemin ambigu- [ ] Si `created_at` contient une heure → colonne `date_created` créée et utilisée pour la relation### Mesures DAX créées- [ ] 10 mesures KPIs de base- [ ] 3 mesures SLA avancés (Breach strict, Dépassement, Ratio)- [ ] 3 mesures Variations vs N-1 avec flèches ▲ ▼- [ ] 2 mesures Alerte contractuelle (Nb cat + texte dynamique)- [ ] 6 mesures ML (4 niveaux + Score moyen + Recall)- [ ] 1 mesure Couleur Barre Breach (formatage conditionnel)- [ ] 4 mesures Agents (Rank, Ecart, Segment, Nb tickets)### Valeurs attendues sans filtre (période complète 2022-2024)| Mesure | Valeur ||---|---|| `[Total Tickets]` | 15 287 || `[Taux SLA Breach]` | 47,3% || `[Taux Backlog]` | 7,4% || `[Nb Tickets Backlog]` | 1 126 || `[First Response Moy]` | 36,8h || `[CSAT Moyen]` | 4,12 || `[Taux Reouverture]` | 7,7% || `[Nb Reouvertures]` | 1 181 || `[Breach SLA Strict]` | 42,9% || `[Depassement Moyen h]` | +121h || `[Ratio SLA Moyen]` | 1,00× || `[Tickets Risque Rouge]` | 314 || `[Tickets Risque Orange]` | 1 542 || `[Tickets Risque Jaune]` | 931 || `[Tickets Risque Vert]` | 271 || `[Score Risque Moy]` | 0,50 || `[Recall Modele]` | 79,6% || `[Nb Categories >50% Breach]` | 3 |### Valeurs catégorie leader- [ ] Demande info breach = 68,6% (rouge)- [ ] Facturation breach = 59,4% (rouge)- [ ] Remboursement breach = 52,6% (rouge)- [ ] Accès compte breach = 20,7% (vert)### Valeurs pays leader- [ ] CI : ~4 000 tickets- [ ] Sénégal : ~3 000 tickets### Valeurs agents critiques- [ ] Moussa Kone breach = 60,1% · 66 alertes ROUGE- [ ] Ramatou Diaby breach = 55,1% · 52 alertes ROUGE- [ ] Kofi Mensah breach = 53,6%### Navigation & slicers- [ ] 4 onglets horizontaux présents sur les 4 pages- [ ] Couleur active distincte par page (violet/orange/vert/rouge)- [ ] Slicer date range (01/01/2022 - 31/12/2024) présent sur les 4 pages- [ ] Slicers synchronisés### Pages- [ ] Page 1 Overview : 6 KPI + courbe SLA + donut statuts + bar catégories + bar pays + alerte contractuelle- [ ] Page 2 Performance SLA : 4 KPI + jauge + bar catégories + donut + heatmap matrix + bar canaux- [ ] Page 3 Agents : Tableau 12 agents + scatter Breach×CSAT coloré par Tier- [ ] Page 4 Alertes ML : 5 KPI + tableau tickets + bar distribution + bar top agents + jauge score

---## 18. Storytelling — Ordre de présentation (5 minutes)### Séquence narrative pour M. KouaméStructure **Problème → Cause → Solution → Impact**, chaque chiffre suivi de sa signification métier.#### 1. Overview — *"Voici l'état global du support"* (1 min)> *"15 287 tickets traités (+24,9% vs l'an dernier). Le taux de SLA breach est de 47,3% — stable vs N-1. L'objectif sectoriel est 30%. On est donc 17 points au-dessus et ce n'est pas une crise ponctuelle. Le CSAT se maintient à 4,12/5, mais le backlog représente 7,4% (1 126 tickets) et 7,7% des tickets sont réouverts (1 181 tickets). Bandeau d'alerte : **3 catégories dépassent 50% de breach** → risque de pénalités contractuelles."*#### 2. Performance SLA — *"Où explose le SLA ?"* (1 min)> *"Sur les tickets en breach, le dépassement moyen est de +121h. Les catégories Billing (Demande info 68,6%, Facturation 59,4%, Remboursement 52,6%) concentrent l'essentiel des violations. La heatmap mensuelle confirme : ces catégories sont rouges quasiment toute l'année. En revanche, les 5 canaux sont homogènes à ~47% — le canal n'est PAS un facteur différenciant. C'est donc un problème de **traitement par catégorie**, pas de canal."*#### 3. Agents — *"Qui performe et qui nécessite un accompagnement ?"* (1 min)> *"Sur 12 agents, **3 Tier 1 sont en alerte critique** : Moussa Kone 60,1%, Ramatou Diaby 55,1%, Kofi Mensah 53,6% de breach. Leur CSAT est aussi le plus bas (3,5 à 3,8). Les Tier 3 (Jean-Marc Dubois, Olivier Martin) affichent les meilleurs CSAT (4,6-4,8) avec un breach sous 45%. Écart de performance = 20 points de breach et 1,3 point de CSAT → il y a un vrai sujet de **formation/coaching Tier 1**."*#### 4. Alertes ML — *"La solution préventive est déployée"* (1 min)> *"Le modèle ML (Recall 79,6%, objectif > 75% ✓) a identifié **314 tickets ROUGE** (intervention immédiate), **1 542 ORANGE** (à surveiller) et **931 JAUNE** (attention). Score de risque moyen 0,50. Les 5 agents Tier 1 déjà identifiés concentrent 214 des 314 alertes ROUGE. En croisant avec leur performance SLA, on a un pattern clair : **ces agents génèrent plus de tickets à risque ET les traitent moins bien**. Le modèle permet au superviseur de basculer le ticket vers un Tier 3 dès sa création."*#### 5. Plan d'action (1 min)1. **Coaching 3 agents Tier 1** sous 15 jours → impact estimé -7 points de breach global2. **Déploiement ML en production** sur dashboard superviseur → évite 80% des breaches futurs par intervention préventive3. **Renégociation SLA Billing** (3 catégories) sous 60 jours → -8 points de breach mécaniquement**Objectif 90 jours : ramener le breach de 47% à 25%.**---> L'apprenant doit pouvoir répondre à la question : **"Que doit faire AfriCare dans les 90 prochains jours ?"**> La réponse est contenue dans les 4 pages du dashboard — pas ailleurs.

---**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.